# ImageJ Plugin Java Kernel Check

Java-only notebook to validate the Jupyter Java kernel and run quick diagnostics for the ImageJ plugin project.

## Prerequisites (strict)
- `java` must be on `PATH`
- `javac` must be on `PATH`
- `mvn` (Apache Maven) must be on `PATH`

This notebook now hard-fails preflight if any of those are missing.
It will also:
1. Parse `FFT/pom.xml` and list direct dependencies
2. Check whether dependency jars are present in local Maven repo (`~/.m2/repository`)
3. Resolve missing dependencies (`mvn dependency:resolve`)
4. Run Maven dependency/plugin update audit (`versions:display-*`)
5. Skip the long build check by default unless explicitly enabled in the final cell

In [10]:
System.out.println("Java kernel is running ✅");
System.out.println("java.version: " + System.getProperty("java.version"));
System.out.println("java.vendor:  " + System.getProperty("java.vendor"));
System.out.println("java.home:    " + System.getProperty("java.home"));
System.out.println("user.dir:     " + System.getProperty("user.dir"));

Java kernel is running ✅
java.version: 25.0.2
java.vendor:  Eclipse Adoptium
java.home:    C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot
user.dir:     c:\Users\dunnmk\repos\imgjplugin\notebooks


In [11]:
import java.nio.file.*;

Path findProjectRoot(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 8 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Path root = findProjectRoot(Paths.get(System.getProperty("user.dir")));
if (root == null) {
    throw new RuntimeException("Could not find project root containing FFT/pom.xml from user.dir");
}

System.out.println("Project root: " + root);
System.out.println("FFT pom.xml exists: " + Files.exists(root.resolve("FFT").resolve("pom.xml")));

Path pluginDir = root.resolve("FFT").resolve("src").resolve("fftanalysis").resolve("imagej");
System.out.println("Plugin source dir: " + pluginDir);
System.out.println("Plugin source dir exists: " + Files.isDirectory(pluginDir));

Project root: c:\Users\dunnmk\repos\imgjplugin
FFT pom.xml exists: true
Plugin source dir: c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej
Plugin source dir exists: true


In [12]:
import java.util.*;

List<String> expected = Arrays.asList(
    "FIBA_Tile_Montage.java",
    "FIBA_Orientation.java",
    "FIBA_Orientation_Profile.java",
    "fibaMain.java",
    "FibaMatlabProcessor.java"
);

Path pluginDir2 = root.resolve("FFT").resolve("src").resolve("fftanalysis").resolve("imagej");
for (String f : expected) {
    Path p = pluginDir2.resolve(f);
    System.out.println((Files.exists(p) ? "OK   " : "MISS ") + p);
}

OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FIBA_Tile_Montage.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FIBA_Orientation.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FIBA_Orientation_Profile.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\fibaMain.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FibaMatlabProcessor.java


In [15]:
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;
import java.util.regex.Pattern;
import javax.xml.parsers.*;
import org.w3c.dom.*;

Path findProjectRoot(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 10 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Path findOnPath(String exe) {
    String path = System.getenv("PATH");
    if (path == null || path.isBlank()) return null;

    String sep = File.pathSeparator;
    String[] pathExts;
    if (System.getProperty("os.name").toLowerCase().contains("win")) {
        String pe = System.getenv("PATHEXT");
        pathExts = (pe == null || pe.isBlank()) ? new String[] {".EXE", ".CMD", ".BAT"} : pe.split(";");
    } else {
        pathExts = new String[] {""};
    }

    for (String part : path.split(Pattern.quote(sep))) {
        if (part == null || part.isBlank()) continue;
        Path base = Paths.get(part.trim());
        if (!Files.isDirectory(base)) continue;

        Path direct = base.resolve(exe);
        if (Files.isRegularFile(direct)) return direct;

        for (String ext : pathExts) {
            String e = ext == null ? "" : ext.trim();
            if (!e.isEmpty() && !exe.toLowerCase().endsWith(e.toLowerCase())) {
                Path withExt = base.resolve(exe + e);
                if (Files.isRegularFile(withExt)) return withExt;
            }
        }
    }
    return null;
}

Path findMavenExecutable() {
    Path p = findOnPath("mvn");
    if (p != null) return p;

    String mavenHome = System.getenv("MAVEN_HOME");
    if (mavenHome != null && !mavenHome.isBlank()) {
        Path bin = Paths.get(mavenHome, "bin");
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }

    List<Path> roots = Arrays.asList(
        Paths.get(System.getProperty("user.home"), "tools", "maven", "apache-maven-3.9.6", "bin"),
        Paths.get(System.getProperty("user.home"), "tools", "apache-maven-3.9.6", "bin"),
        Paths.get(System.getProperty("user.home"), ".tools", "apache-maven-3.9.6", "bin")
    );
    for (Path bin : roots) {
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }
    return null;
}

String runAndCollect(List<String> cmd, Path cwd, int timeoutSeconds, int maxLines) throws Exception {
    ProcessBuilder pb = new ProcessBuilder(cmd);
    if (cwd != null) pb.directory(cwd.toFile());
    pb.redirectErrorStream(true);

    Process proc = pb.start();
    StringBuilder sb = new StringBuilder();
    int shown = 0;

    try (BufferedReader br = new BufferedReader(new InputStreamReader(proc.getInputStream(), StandardCharsets.UTF_8))) {
        String line;
        long deadline = System.currentTimeMillis() + timeoutSeconds * 1000L;
        while (true) {
            while (br.ready() && (line = br.readLine()) != null) {
                if (shown < maxLines) {
                    sb.append(line).append("\n");
                    shown++;
                }
            }

            if (!proc.isAlive()) break;
            if (System.currentTimeMillis() > deadline) {
                proc.destroyForcibly();
                throw new RuntimeException("Command timed out after " + timeoutSeconds + "s: " + String.join(" ", cmd));
            }
            Thread.sleep(100);
        }

        while ((line = br.readLine()) != null) {
            if (shown < maxLines) {
                sb.append(line).append("\n");
                shown++;
            }
        }
    }

    int exit = proc.waitFor();
    sb.append("[exit=").append(exit).append("]\n");
    if (exit != 0) {
        throw new RuntimeException("Command failed (exit=" + exit + "): " + String.join(" ", cmd) + "\n" + sb);
    }
    return sb.toString();
}

String childText(Element parent, String tag) {
    NodeList nl = parent.getElementsByTagName(tag);
    if (nl.getLength() == 0) return null;
    return nl.item(0).getTextContent().trim();
}

String resolveProps(String in, Map<String, String> props) {
    if (in == null) return null;
    String out = in;
    int guard = 0;
    while (out.contains("${") && guard++ < 20) {
        int s = out.indexOf("${");
        int e = out.indexOf("}", s + 2);
        if (s < 0 || e < 0) break;
        String key = out.substring(s + 2, e);
        String val = props.getOrDefault(key, "${" + key + "}");
        out = out.substring(0, s) + val + out.substring(e + 1);
    }
    return out;
}

Path root2 = findProjectRoot(Paths.get(System.getProperty("user.dir")));
if (root2 == null) throw new RuntimeException("Could not find project root containing FFT/pom.xml from user.dir");
Path fftDir = root2.resolve("FFT");
Path pom = fftDir.resolve("pom.xml");
if (!Files.exists(pom)) throw new RuntimeException("Missing pom.xml at: " + pom);

System.out.println("Project root: " + root2);
System.out.println("FFT dir:      " + fftDir);
System.out.println("pom.xml:      " + pom);

Path javaPath = findOnPath("java");
Path javacPath = findOnPath("javac");
Path mvnPath = findMavenExecutable();

System.out.println("\nPATH checks:");
System.out.println("java  -> " + (javaPath == null ? "MISSING" : javaPath));
System.out.println("javac -> " + (javacPath == null ? "MISSING" : javacPath));
System.out.println("mvn   -> " + (mvnPath == null ? "MISSING" : mvnPath));

boolean toolsOk = (javaPath != null && javacPath != null && mvnPath != null);
if (!toolsOk) {
    System.out.println("\nPreflight blocked: missing required executable(s). Ensure java/javac/mvn are discoverable.");
} else {
    System.out.println("\nVersion checks:");
    System.out.println(runAndCollect(Arrays.asList(javaPath.toString(), "-version"), fftDir, 30, 30));
    System.out.println(runAndCollect(Arrays.asList(javacPath.toString(), "-version"), fftDir, 30, 30));
    System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-v"), fftDir, 45, 80));

    DocumentBuilderFactory dbf = DocumentBuilderFactory.newInstance();
    dbf.setNamespaceAware(false);
    DocumentBuilder db = dbf.newDocumentBuilder();
    Document doc = db.parse(pom.toFile());

    Map<String, String> props = new LinkedHashMap<>();
    NodeList propNodes = doc.getElementsByTagName("properties");
    if (propNodes.getLength() > 0) {
        Node n = propNodes.item(0);
        NodeList kids = n.getChildNodes();
        for (int i = 0; i < kids.getLength(); i++) {
            Node c = kids.item(i);
            if (c.getNodeType() == Node.ELEMENT_NODE) props.put(c.getNodeName(), c.getTextContent().trim());
        }
    }

    NodeList depNodes = doc.getElementsByTagName("dependency");
    List<String> deps = new ArrayList<>();
    for (int i = 0; i < depNodes.getLength(); i++) {
        Element d = (Element) depNodes.item(i);
        String g = resolveProps(childText(d, "groupId"), props);
        String a = resolveProps(childText(d, "artifactId"), props);
        String v = resolveProps(childText(d, "version"), props);
        String s = resolveProps(childText(d, "scope"), props);
        if (s == null) s = "compile";
        if (g != null && a != null && v != null) deps.add(g + ":" + a + ":" + v + ":" + s);
    }

    System.out.println("Direct dependencies in pom.xml:");
    for (String d : deps) System.out.println("  - " + d);

    Path m2 = Paths.get(System.getProperty("user.home"), ".m2", "repository");
    System.out.println("\nLocal Maven repo: " + m2);
    System.out.println("\nLocal dependency presence (jar):");
    for (String d : deps) {
        String[] p = d.split(":");
        String g = p[0], a = p[1], v = p[2];
        Path jar = m2.resolve(g.replace('.', File.separatorChar)).resolve(a).resolve(v).resolve(a + "-" + v + ".jar");
        System.out.println((Files.isRegularFile(jar) ? "FOUND " : "MISS  ") + d + " -> " + jar);
    }

    System.out.println("\nResolving Maven dependencies now (downloads if missing)...");
    System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-B", "-DskipTests", "dependency:resolve"), fftDir, 600, 250));

    System.out.println("Checking for dependency/plugin updates...");
    System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-B", "versions:display-dependency-updates", "versions:display-plugin-updates"), fftDir, 600, 350));

    System.out.println("\nPreflight complete ✅");
    System.out.println("No build/test command has been run in this cell.");

    boolean RUN_MAVEN_BUILD_CHECK = false;
    if (RUN_MAVEN_BUILD_CHECK) {
        System.out.println("\nRunning bounded Maven build check...");
        System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-B", "-DskipTests", "test"), fftDir, 900, 400));
        System.out.println("Build check finished successfully ✅");
    } else {
        System.out.println("\nBuild check skipped by design (RUN_MAVEN_BUILD_CHECK=false).");
    }
}

Project root: c:\Users\dunnmk\repos\imgjplugin
FFT dir:      c:\Users\dunnmk\repos\imgjplugin\FFT
pom.xml:      c:\Users\dunnmk\repos\imgjplugin\FFT\pom.xml

PATH checks:
java  -> C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot\bin\java.EXE
javac -> C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot\bin\javac.EXE
mvn   -> C:\Users\dunnmk\tools\maven\apache-maven-3.9.6\bin\mvn.cmd

Version checks:
openjdk version "25.0.2" 2026-01-20 LTS
OpenJDK Runtime Environment Temurin-25.0.2+10 (build 25.0.2+10-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.2+10 (build 25.0.2+10-LTS, mixed mode, sharing)
[exit=0]

javac 25.0.2
[exit=0]


Apache Maven 3.9.6 (bc0240f3c744dd6b6ec2920b3cd08dcc295161ae)
Maven home: C:\Users\dunnmk\tools\maven\apache-maven-3.9.6
Java version: 25.0.2, vendor: Eclipse Adoptium, runtime: C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot
Default locale: en_US, platform encoding: UTF-8
OS name: "w

## Visualization pipeline (dual-image comparison)

This section runs the Java pipeline for **two input images** and creates two final figure outputs so orientation direction can be compared directly.

### Default quality enhancement (STORM-like)
Before running the pipeline cell, run the preprocessing cell to create **STORM-like cleaned images** (bandpass + spot enhancement).
The pipeline cell now enforces these STORM-like preprocessed images by default.

Inputs:
1. `C:/Users/dunnmk/Downloads/C15D5P001 (1).jpg`
2. `C:/Users/dunnmk/OneDrive - Michigan Medicine/Pictures/Picture1.jpg`

Outputs are written to separate folders under `notebooks/_assets/fiba_tile_montage/` to avoid overlap.

In [36]:
import java.awt.image.BufferedImage;
import java.nio.file.*;
import java.util.*;
import javax.imageio.ImageIO;

// -----------------------------------
// STORM-like preprocessing (default)
// -----------------------------------
final double SIGMA_SMALL = 1.0;
final double SIGMA_LARGE = 3.0;
final double THRESH_STD = 0.50;
final double GAMMA = 0.70;

double[][] toGray(BufferedImage img) {
    int h = img.getHeight();
    int w = img.getWidth();
    double[][] g = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int gg = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            g[y][x] = 0.299 * r + 0.587 * gg + 0.114 * b;
        }
    }
    return g;
}

BufferedImage fromGray(double[][] g) {
    int h = g.length;
    int w = g[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(Math.max(0, Math.min(255, g[y][x])));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

double[][] copy2D(double[][] a) {
    int h = a.length, w = a[0].length;
    double[][] b = new double[h][w];
    for (int y = 0; y < h; y++) System.arraycopy(a[y], 0, b[y], 0, w);
    return b;
}

void normalize01(double[][] a) {
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    for (int y = 0; y < a.length; y++) {
        for (int x = 0; x < a[0].length; x++) {
            a[y][x] = (a[y][x] - min) / span;
        }
    }
}

double[] gaussianKernel1D(double sigma) {
    int radius = Math.max(1, (int)Math.ceil(3.0 * sigma));
    int n = radius * 2 + 1;
    double[] k = new double[n];
    double sum = 0.0;
    for (int i = -radius; i <= radius; i++) {
        double v = Math.exp(-(i * i) / (2.0 * sigma * sigma));
        k[i + radius] = v;
        sum += v;
    }
    for (int i = 0; i < n; i++) k[i] /= sum;
    return k;
}

double[][] convolveHorizontal(double[][] src, double[] k) {
    int h = src.length, w = src[0].length;
    int r = k.length / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int i = -r; i <= r; i++) {
                int xx = Math.min(w - 1, Math.max(0, x + i));
                s += src[y][xx] * k[i + r];
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] convolveVertical(double[][] src, double[] k) {
    int h = src.length, w = src[0].length;
    int r = k.length / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int i = -r; i <= r; i++) {
                int yy = Math.min(h - 1, Math.max(0, y + i));
                s += src[yy][x] * k[i + r];
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] gaussianBlur(double[][] src, double sigma) {
    double[] k = gaussianKernel1D(sigma);
    return convolveVertical(convolveHorizontal(src, k), k);
}

double[][] stormLikeEnhance(double[][] gray255) {
    double[][] norm = copy2D(gray255);
    normalize01(norm);

    double[][] gSmall = gaussianBlur(norm, SIGMA_SMALL);
    double[][] gLarge = gaussianBlur(norm, SIGMA_LARGE);

    int h = norm.length, w = norm[0].length;
    double[][] dog = new double[h][w];
    double mean = 0.0;
    double sq = 0.0;
    int n = h * w;

    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            dog[y][x] = gSmall[y][x] - gLarge[y][x];
            mean += dog[y][x];
            sq += dog[y][x] * dog[y][x];
        }
    }
    mean /= n;
    double var = Math.max(0.0, (sq / n) - mean * mean);
    double std = Math.sqrt(var);
    double thresh = mean + THRESH_STD * std;

    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double v = Math.max(0.0, dog[y][x] - thresh);
            v = Math.pow(v, GAMMA);
            out[y][x] = v;
        }
    }

    // Rescale to 0..255
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : out) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            out[y][x] = 255.0 * (out[y][x] - min) / span;
        }
    }
    return out;
}

Path rootPre = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootPre == null) throw new RuntimeException("Could not locate project root for preprocessing.");
Path preDir = rootPre.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("preprocessed");
Files.createDirectories(preDir);

List<Path> rawInputs = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> rawBases = Arrays.asList("C15D5P001_1", "Picture1");

System.out.println("Running STORM-like preprocessing...");
System.out.println("SIGMA_SMALL=" + SIGMA_SMALL + ", SIGMA_LARGE=" + SIGMA_LARGE + ", THRESH_STD=" + THRESH_STD + ", GAMMA=" + GAMMA);

for (int i = 0; i < rawInputs.size(); i++) {
    Path in = rawInputs.get(i);
    if (!Files.isRegularFile(in)) throw new RuntimeException("Missing input for preprocessing: " + in);

    BufferedImage img = ImageIO.read(in.toFile());
    if (img == null) throw new RuntimeException("Could not read image: " + in);

    double[][] gray = toGray(img);
    double[][] enhanced = stormLikeEnhance(gray);

    Path out = preDir.resolve(rawBases.get(i) + "_clean_storm.jpg");
    ImageIO.write(fromGray(enhanced), "jpg", out.toFile());
    System.out.println("Preprocessed image written: " + out);
}

System.out.println("\nPreprocessing complete ✅ (STORM-like bandpass + spot enhancement)");

Running STORM-like preprocessing...
SIGMA_SMALL=1.0, SIGMA_LARGE=3.0, THRESH_STD=0.5, GAMMA=0.7
Preprocessed image written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\preprocessed\C15D5P001_1_clean_storm.jpg
Preprocessed image written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\preprocessed\Picture1_clean_storm.jpg

Preprocessing complete ✅ (STORM-like bandpass + spot enhancement)


In [40]:
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;
import java.util.regex.Matcher;
import java.util.regex.Pattern;

Path findProjectRootViz(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Path findOnPathViz(String exe) {
    String path = System.getenv("PATH");
    if (path == null || path.isBlank()) return null;
    String sep = File.pathSeparator;
    String[] exts = System.getProperty("os.name").toLowerCase().contains("win")
        ? new String[] {"", ".cmd", ".bat", ".exe"}
        : new String[] {""};

    for (String part : path.split(Pattern.quote(sep))) {
        if (part == null || part.isBlank()) continue;
        Path dir = Paths.get(part.trim());
        if (!Files.isDirectory(dir)) continue;
        for (String ext : exts) {
            Path cand = dir.resolve(exe + ext);
            if (Files.isRegularFile(cand)) return cand;
        }
    }
    return null;
}

Path findMavenViz() {
    Path p = findOnPathViz("mvn");
    if (p != null) return p;
    String mh = System.getenv("MAVEN_HOME");
    if (mh != null && !mh.isBlank()) {
        Path bin = Paths.get(mh, "bin");
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }
    for (Path bin : Arrays.asList(
            Paths.get(System.getProperty("user.home"), "tools", "maven", "apache-maven-3.9.6", "bin"),
            Paths.get(System.getProperty("user.home"), "tools", "apache-maven-3.9.6", "bin"),
            Paths.get(System.getProperty("user.home"), ".tools", "apache-maven-3.9.6", "bin")
    )) {
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }
    return null;
}

String runAndCollectViz(List<String> cmd, Path cwd, int timeoutSeconds, int maxLines) throws Exception {
    ProcessBuilder pb = new ProcessBuilder(cmd);
    pb.directory(cwd.toFile());
    pb.redirectErrorStream(true);
    Process proc = pb.start();

    StringBuilder sb = new StringBuilder();
    int shown = 0;
    long deadline = System.currentTimeMillis() + timeoutSeconds * 1000L;
    try (BufferedReader br = new BufferedReader(new InputStreamReader(proc.getInputStream(), StandardCharsets.UTF_8))) {
        String line;
        while (true) {
            while (br.ready() && (line = br.readLine()) != null) {
                if (shown < maxLines) {
                    sb.append(line).append("\n");
                    shown++;
                }
            }
            if (!proc.isAlive()) break;
            if (System.currentTimeMillis() > deadline) {
                proc.destroyForcibly();
                throw new RuntimeException("Pipeline command timed out: " + String.join(" ", cmd));
            }
            Thread.sleep(100);
        }
        while ((line = br.readLine()) != null) {
            if (shown < maxLines) {
                sb.append(line).append("\n");
                shown++;
            }
        }
    }
    int exit = proc.waitFor();
    sb.append("[exit=").append(exit).append("]\n");
    if (exit != 0) throw new RuntimeException("Command failed: " + String.join(" ", cmd) + "\n" + sb);
    return sb.toString();
}

Path rootViz = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootViz == null) throw new RuntimeException("Could not locate project root for visualization step.");
Path fftDirViz = rootViz.resolve("FFT");

Path mvnViz = findMavenViz();
if (mvnViz == null) {
    throw new RuntimeException("Maven executable not found, cannot run pipeline generation step.");
}

Path preDir = rootViz.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("preprocessed");
boolean ENFORCE_STORM_PREPROCESSED_DEFAULT = true;

List<Path> inputImages = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> baseNames = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> outputDirs = Arrays.asList(
    rootViz.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootViz.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

for (int idx = 0; idx < inputImages.size(); idx++) {
    Path rawImage = inputImages.get(idx);
    String baseNameViz = baseNames.get(idx);
    Path assetsDirViz = outputDirs.get(idx);

    Path cleanedCandidate = preDir.resolve(baseNameViz + "_clean_storm.jpg");
    Path sourceImageViz = rawImage;
    if (ENFORCE_STORM_PREPROCESSED_DEFAULT) {
        if (!Files.isRegularFile(cleanedCandidate)) {
            throw new RuntimeException("Default mode requires STORM-like preprocessed image, but file is missing: " + cleanedCandidate + "\nRun the preprocessing cell first.");
        }
        sourceImageViz = cleanedCandidate;
    }

    if (!Files.isRegularFile(sourceImageViz)) {
        throw new RuntimeException("Source image not found: " + sourceImageViz + "\nCannot run real pipeline without this file.");
    }
    Files.createDirectories(assetsDirViz);

    System.out.println("\n==================================================");
    System.out.println("Running real tile pipeline from source image...");
    System.out.println("Case:   " + baseNameViz);
    System.out.println("Raw:    " + rawImage);
    System.out.println("Using:  " + sourceImageViz + " (STORM default enforced)");
    System.out.println("Output: " + assetsDirViz);

    List<String> cmd = Arrays.asList(
        mvnViz.toString(),
        "-B",
        "-Dtest=fftanalysis.imagej.GenerateTileMontageFromSourceTest",
        "-Dfiba.input=" + sourceImageViz.toString(),
        "-Dfiba.output=" + assetsDirViz.toString(),
        "-Dfiba.base=" + baseNameViz,
        "-Dfiba.tilesY=10",
        "test"
    );
    System.out.println(runAndCollectViz(cmd, fftDirViz, 900, 220));

    Path boxesPathViz = assetsDirViz.resolve(baseNameViz + "_tile_boxes.jpg");
    Path stackPathViz = assetsDirViz.resolve(baseNameViz + "_tile_montage.jpg");
    Path csvPathViz = assetsDirViz.resolve(baseNameViz + "_tile_results.csv");
    if (!Files.isRegularFile(boxesPathViz) || !Files.isRegularFile(stackPathViz) || !Files.isRegularFile(csvPathViz)) {
        throw new RuntimeException("Pipeline run completed but expected outputs are missing in: " + assetsDirViz);
    }

    Pattern tilePatternViz = Pattern.compile("^" + Pattern.quote(baseNameViz) + "_tile(\\d+)_crop\\.jpg$");
    Set<Integer> idsViz = new LinkedHashSet<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(assetsDirViz, "*_crop.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePatternViz.matcher(p.getFileName().toString());
            if (m.matches()) idsViz.add(Integer.parseInt(m.group(1)));
        }
    }
    List<Integer> tileIdsViz = new ArrayList<>(idsViz);
    tileIdsViz.sort(Comparator.naturalOrder());
    if (tileIdsViz.isEmpty()) throw new RuntimeException("No generated crop tiles found after pipeline run for " + baseNameViz);

    System.out.println("Generated assets confirmed for " + baseNameViz + ":");
    System.out.println("Tiles: " + tileIdsViz);
    System.out.println("Overlay: " + boxesPathViz.getFileName());
    System.out.println("Tile stack: " + stackPathViz.getFileName());
    System.out.println("CSV: " + csvPathViz.getFileName());
}

System.out.println("\nDual-image pipeline generation complete ✅");


Running real tile pipeline from source image...
Case:   C15D5P001_1
Raw:    C:\Users\dunnmk\Downloads\C15D5P001 (1).jpg
Using:  c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\preprocessed\C15D5P001_1_clean_storm.jpg (STORM default enforced)
Output: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source

[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.mikdunn:imgjplugin-fft >---------------------
[INFO] Building ImageJ FFT Orientation Plugin 0.1.0-SNAPSHOT
[INFO]   from pom.xml
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- enforcer:3.6.1:enforce (enforce-maven) @ imgjplugin-fft ---
[INFO] Rule 0: org.apache.maven.enforcer.rules.version.RequireMavenVersion passed
[INFO] 
[INFO] --- resources:3.3.1:resources (default-resources) @ imgjplugin-fft ---
[INFO] Copying 1 resource from resources to target\classes
[INFO] 
[INFO] --- compiler:3.15.0:compile (def

In [39]:
import java.awt.*;
import java.awt.geom.Rectangle2D;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;

BufferedImage readOrPlaceholder(Path p, int w, int h, String label) throws IOException {
    if (Files.exists(p)) {
        BufferedImage img = ImageIO.read(p.toFile());
        if (img != null) return img;
    }
    BufferedImage miss = new BufferedImage(w, h, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = miss.createGraphics();
    g.setColor(new Color(245, 245, 245));
    g.fillRect(0, 0, w, h);
    g.setColor(new Color(200, 60, 60));
    g.setFont(new Font("SansSerif", Font.BOLD, 18));
    g.drawString("MISSING", 16, 30);
    g.setColor(Color.DARK_GRAY);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString(label, 16, 54);
    g.dispose();
    return miss;
}

void drawFit(Graphics2D g, BufferedImage src, int x, int y, int w, int h) {
    double sx = w / (double) src.getWidth();
    double sy = h / (double) src.getHeight();
    double s = Math.min(sx, sy);
    int nw = Math.max(1, (int) Math.round(src.getWidth() * s));
    int nh = Math.max(1, (int) Math.round(src.getHeight() * s));
    int ox = x + (w - nw) / 2;
    int oy = y + (h - nh) / 2;
    g.drawImage(src, ox, oy, nw, nh, null);
}

int drawSectionTitle(Graphics2D g, String title, int x, int y, int w) {
    g.setColor(new Color(24, 44, 74));
    g.setFont(new Font("SansSerif", Font.BOLD, 22));
    g.drawString(title, x, y + 22);
    g.setColor(new Color(220, 226, 235));
    g.fill(new Rectangle2D.Double(x, y + 28, w, 2));
    return y + 36;
}

int drawSingleImageSection(Graphics2D g, String title, Path p, int x, int y, int w, int h) throws IOException {
    int yy = drawSectionTitle(g, title, x, y, w);
    g.setColor(new Color(232, 232, 232));
    g.fillRect(x - 1, yy - 1, w + 2, h + 2);
    BufferedImage img = readOrPlaceholder(p, w, h, p.getFileName().toString());
    drawFit(g, img, x, yy, w, h);
    return yy + h + 18;
}

int drawGridSection(Graphics2D g, String title, List<Integer> ids, String key, Path assetsDir, String baseName, int x, int y, int columns, int cellW, int cellH, int gap) throws IOException {
    int yy = drawSectionTitle(g, title, x, y, columns * cellW + (columns - 1) * gap);
    int rows = (int) Math.ceil(ids.size() / (double) columns);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));

    for (int i = 0; i < ids.size(); i++) {
        int id = ids.get(i);
        int r = i / columns;
        int c = i % columns;
        int cx = x + c * (cellW + gap);
        int cy = yy + r * (cellH + 26 + gap);

        Path p = assetsDir.resolve(baseName + "_tile" + id + "_" + key + ".jpg");
        BufferedImage img = readOrPlaceholder(p, cellW, cellH, p.getFileName().toString());

        g.setColor(new Color(30, 30, 30));
        g.drawString("Tile " + id, cx + 4, cy + 14);
        g.setColor(new Color(232, 232, 232));
        g.fillRect(cx - 1, cy + 17 - 1, cellW + 2, cellH + 2);
        drawFit(g, img, cx, cy + 17, cellW, cellH);
    }

    return yy + rows * (cellH + 26 + gap) + 8;
}

Path rootViz2 = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootViz2 == null) throw new RuntimeException("Could not locate project root for final figure composition.");

List<Path> inputImages2 = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> baseNames2 = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> outputDirs2 = Arrays.asList(
    rootViz2.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootViz2.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

for (int caseIdx = 0; caseIdx < baseNames2.size(); caseIdx++) {
    String baseNameViz2 = baseNames2.get(caseIdx);
    Path assetsDirViz2 = outputDirs2.get(caseIdx);
    Path originalPath = inputImages2.get(caseIdx);

    Path tileStackPath = assetsDirViz2.resolve(baseNameViz2 + "_tile_montage.jpg");
    Path tileBoxesPath = assetsDirViz2.resolve(baseNameViz2 + "_tile_boxes.jpg");
    if (!Files.isRegularFile(originalPath)) throw new RuntimeException("Missing source image: " + originalPath);
    if (!Files.isRegularFile(tileStackPath)) throw new RuntimeException("Missing generated tile stack: " + tileStackPath + "\nRun previous pipeline cell first.");
    if (!Files.isRegularFile(tileBoxesPath)) throw new RuntimeException("Missing generated tile boxes output: " + tileBoxesPath + "\nRun previous pipeline cell first.");

    Pattern tilePatternViz2 = Pattern.compile("^" + Pattern.quote(baseNameViz2) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> tileIdsViz2 = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(assetsDirViz2, "*_crop.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePatternViz2.matcher(p.getFileName().toString());
            if (m.matches()) tileIdsViz2.add(Integer.parseInt(m.group(1)));
        }
    }
    tileIdsViz2.sort(Comparator.naturalOrder());
    if (tileIdsViz2.isEmpty()) throw new RuntimeException("No generated tile images found in " + assetsDirViz2);

    List<Integer> firstTen = tileIdsViz2.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = tileIdsViz2;

    int margin = 28;
    int pageW = 1320;
    int usableW = pageW - 2 * margin;
    int columns = 5;
    int gap = 14;
    int cellW = (usableW - (columns - 1) * gap) / columns;
    int cellH = 165;
    int rows = (int) Math.ceil(firstTen.size() / (double) columns);
    int gridH = rows * (cellH + 26 + gap) + 8;

    int hOriginal = 380;
    int hTileStack = 330;
    int titleTop = 62;
    int bottomPad = 28;
    int sectionGap = 8;

    int pageH = titleTop
        + (36 + hOriginal + 18)
        + sectionGap
        + (36 + hTileStack + 18)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + bottomPad;

    BufferedImage canvas = new BufferedImage(pageW, pageH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = canvas.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setRenderingHint(RenderingHints.KEY_INTERPOLATION, RenderingHints.VALUE_INTERPOLATION_BILINEAR);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, pageW, pageH);

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 30));
    g.drawString("Final FIBA Figure (Generated from Source): " + baseNameViz2, margin, 38);
    g.setFont(new Font("SansSerif", Font.PLAIN, 14));
    g.drawString("Order: source image -> tile stack -> images 1-10 -> FFT -> polar -> SOL -> mask -> reconstruction", margin, 56);

    int y = titleTop;
    y = drawSingleImageSection(g, "1) Source image", originalPath, margin, y, usableW, hOriginal);
    y += sectionGap;
    y = drawSingleImageSection(g, "2) Tiles stack (generated by pipeline)", tileStackPath, margin, y, usableW, hTileStack);
    y += sectionGap;
    y = drawGridSection(g, "3) Images numbered 1-10", firstTen, "crop", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "4) FFT image", firstTen, "fft", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "5) Polar coordinates", firstTen, "polar", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "6) SOL graph", firstTen, "sol", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "7) Mask", firstTen, "mask", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "8) Reconstruction", firstTen, "rec", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);

    g.dispose();

    Path outPath = assetsDirViz2.resolve(baseNameViz2 + "_final_figure_java.png");
    ImageIO.write(canvas, "png", outPath.toFile());
    System.out.println("Final figure written: " + outPath);
    System.out.println("Source image used: " + originalPath);
    System.out.println("Tile boxes were generated by pipeline: " + tileBoxesPath + " (exists=" + Files.exists(tileBoxesPath) + ")");
}

System.out.println("\nDual final-figure generation complete ✅");

Final figure written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_final_figure_java.png
Source image used: C:\Users\dunnmk\Downloads\C15D5P001 (1).jpg
Tile boxes were generated by pipeline: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_tile_boxes.jpg (exists=true)
Final figure written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_final_figure_java.png
Source image used: C:\Users\dunnmk\OneDrive - Michigan Medicine\Pictures\Picture1.jpg
Tile boxes were generated by pipeline: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_tile_boxes.jpg (exists=true)

Dual final-figure generation complete ✅


In [43]:
import java.awt.*;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

double maskCoverage(BufferedImage img) {
    int w = img.getWidth();
    int h = img.getHeight();
    long on = 0L;
    long total = (long) w * h;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            int gray = (r + g + b) / 3;
            if (gray > 16) on++;
        }
    }
    return total == 0 ? 0.0 : (100.0 * on / (double) total);
}

BufferedImage readMaskStrict(Path p) throws Exception {
    if (!Files.isRegularFile(p)) {
        throw new RuntimeException("Missing mask image (expected graph input): " + p);
    }
    BufferedImage img = ImageIO.read(p.toFile());
    if (img == null) throw new RuntimeException("Could not decode mask image: " + p);
    return img;
}

Path rootMask = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootMask == null) throw new RuntimeException("Could not locate project root for mask visualization.");

List<String> maskBases = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> maskDirs = Arrays.asList(
    rootMask.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootMask.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

for (int caseIdx = 0; caseIdx < maskBases.size(); caseIdx++) {
    String base = maskBases.get(caseIdx);
    Path dir = maskDirs.get(caseIdx);

    Pattern tilePattern = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_mask\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(dir, "*_mask.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePattern.matcher(p.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) throw new RuntimeException("No mask files found for " + base + " in " + dir);

    List<Integer> firstTen = ids.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = ids;

    int cols = 5, gap = 14, margin = 22;
    int cellW = 210, cellH = 165;
    int rows = (int)Math.ceil(firstTen.size() / (double)cols);
    int pageW = margin * 2 + cols * cellW + (cols - 1) * gap;
    int pageH = 90 + rows * (cellH + 44 + gap) + 28;

    BufferedImage canvas = new BufferedImage(pageW, pageH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = canvas.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, pageW, pageH);

    g.setColor(new Color(20, 36, 68));
    g.setFont(new Font("SansSerif", Font.BOLD, 24));
    g.drawString("Mask panel check: " + base, margin, 34);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString("Each tile is loaded from *_mask.jpg. Coverage shown under tile.", margin, 56);

    double sumCov = 0.0;
    for (int i = 0; i < firstTen.size(); i++) {
        int tile = firstTen.get(i);
        int r = i / cols;
        int c = i % cols;
        int x = margin + c * (cellW + gap);
        int y = 74 + r * (cellH + 44 + gap);

        Path maskPath = dir.resolve(base + "_tile" + tile + "_mask.jpg");
        BufferedImage mask = readMaskStrict(maskPath);
        double cov = maskCoverage(mask);
        sumCov += cov;

        g.setColor(new Color(35, 35, 35));
        g.setFont(new Font("SansSerif", Font.BOLD, 13));
        g.drawString("Tile " + tile, x + 2, y + 14);

        g.setColor(new Color(232, 232, 232));
        g.fillRect(x - 1, y + 18 - 1, cellW + 2, cellH + 2);
        drawFit(g, mask, x, y + 18, cellW, cellH);

        g.setColor(new Color(120, 18, 18));
        g.setFont(new Font("SansSerif", Font.PLAIN, 12));
        g.drawString(String.format(Locale.US, "mask coverage: %.2f%%", cov), x + 2, y + 18 + cellH + 16);
    }

    g.dispose();

    Path out = dir.resolve(base + "_mask_panel_check.png");
    ImageIO.write(canvas, "png", out.toFile());

    double meanCov = sumCov / firstTen.size();
    System.out.printf(Locale.US, "%n%s mask panel written: %s%n", base, out);
    System.out.printf(Locale.US, "%s mean mask coverage over shown tiles: %.2f%%%n", base, meanCov);
}

System.out.println("\nMask graphing verification complete for BOTH images ✅");


C15D5P001_1 mask panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_mask_panel_check.png
C15D5P001_1 mean mask coverage over shown tiles: 10.24%

Picture1 mask panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_mask_panel_check.png
Picture1 mean mask coverage over shown tiles: 16.01%

Mask graphing verification complete for BOTH images ✅


## Mask stage verification (both images)

Run the previous Java cell to generate explicit mask-panel outputs for both datasets:

- `_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_mask_panel_check.png`
- `_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_mask_panel_check.png`

C15 mask panel:

![](_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_mask_panel_check.png)

Picture1 mask panel:

![](_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_mask_panel_check.png)

In [ ]:
import java.nio.file.*;
import java.nio.charset.StandardCharsets;
import java.util.*;

Map<Integer, Double> loadAnglesForComparison(Path csv) throws Exception {
    Map<Integer, Double> out = new LinkedHashMap<>();
    List<String> lines = Files.readAllLines(csv, StandardCharsets.UTF_8);
    if (lines.isEmpty()) return out;

    String[] header = lines.get(0).split(",");
    int idCol = -1;
    int fiberCol = -1;
    int adjCol = -1;
    for (int i = 0; i < header.length; i++) {
        String h = header[i].trim();
        if ("tile_id".equals(h)) idCol = i;
        if ("pAng_fiber_axis".equals(h)) fiberCol = i;
        if ("pAng_adj".equals(h)) adjCol = i;
    }
    if (idCol < 0) {
        throw new RuntimeException("CSV missing required column tile_id: " + csv);
    }
    int useCol = (fiberCol >= 0) ? fiberCol : adjCol;
    if (useCol < 0) {
        throw new RuntimeException("CSV missing angle columns (need pAng_fiber_axis or pAng_adj): " + csv);
    }

    for (int i = 1; i < lines.size(); i++) {
        String line = lines.get(i).trim();
        if (line.isEmpty()) continue;
        String[] cols = line.split(",");
        if (cols.length <= Math.max(idCol, useCol)) continue;
        int id = Integer.parseInt(cols[idCol].trim());
        double a = Double.parseDouble(cols[useCol].trim());
        out.put(id, a);
    }
    return out;
}

double wrap180(double a) {
    double x = a % 180.0;
    if (x < -90.0) x += 180.0;
    if (x > 90.0) x -= 180.0;
    return x;
}

Path rootCmp = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
Path csvA = rootCmp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source").resolve("C15D5P001_1_tile_results.csv");
Path csvB = rootCmp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1").resolve("Picture1_tile_results.csv");

if (!Files.isRegularFile(csvA) || !Files.isRegularFile(csvB)) {
    throw new RuntimeException("Missing comparison CSV(s). Run the previous pipeline cell first.\nA=" + csvA + "\nB=" + csvB);
}

Map<Integer, Double> a = loadAnglesForComparison(csvA);
Map<Integer, Double> b = loadAnglesForComparison(csvB);

Set<Integer> common = new TreeSet<>(a.keySet());
common.retainAll(b.keySet());
if (common.isEmpty()) throw new RuntimeException("No common tile ids between comparison CSV files.");

System.out.println("Orientation comparison (deg)");
System.out.println("A = C15D5P001_1, B = Picture1");
System.out.println("Using pAng_fiber_axis when present, otherwise pAng_adj.");
System.out.println("tile\tA\tB\t(B-A)\taxis_diff_mod180");

double sumDelta = 0.0;
double sumAxisDiff = 0.0;
for (int t : common) {
    double av = a.get(t);
    double bv = b.get(t);
    double delta = bv - av;
    double axisDiff = Math.abs(wrap180(delta));
    sumDelta += delta;
    sumAxisDiff += axisDiff;
    System.out.printf(Locale.US, "%d\t%.1f\t%.1f\t%.1f\t%.1f%n", t, av, bv, delta, axisDiff);
}

double n = common.size();
System.out.printf(Locale.US, "\nMean delta (B-A): %.2f deg%n", sumDelta / n);
System.out.printf(Locale.US, "Mean axis diff mod 180: %.2f deg%n", sumAxisDiff / n);

Orientation comparison (pAng_adj, degrees)
A = C15D5P001_1, B = Picture1
tile	A	B	(B-A)	axis_diff_mod180
1	-25.0	-50.0	-25.0	25.0
2	-25.0	-53.0	-28.0	28.0
3	-22.0	-61.0	-39.0	39.0
4	-23.0	-64.0	-41.0	41.0
5	-7.0	-61.0	-54.0	54.0
6	-2.0	-61.0	-59.0	59.0
7	2.0	-59.0	-61.0	61.0
8	12.0	-62.0	-74.0	74.0
9	17.0	-64.0	-81.0	81.0
10	19.0	-64.0	-83.0	83.0

Mean delta (B-A): -54.50 deg
Mean axis diff mod 180: 54.50 deg
If direction looks perpendicular visually, test +90 deg post-adjustment for display/reporting.


In [35]:
Path rootDisp = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
Path figA = rootDisp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source").resolve("C15D5P001_1_final_figure_java.png");
Path figB = rootDisp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1").resolve("Picture1_final_figure_java.png");

System.out.println("Figure A exists: " + Files.exists(figA) + " -> " + figA);
System.out.println("Figure B exists: " + Files.exists(figB) + " -> " + figB);
if (Files.exists(figB)) {
    System.out.println("Figure B size(bytes): " + Files.size(figB));
}

Figure A exists: true -> c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_final_figure_java.png
Figure B exists: true -> c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_final_figure_java.png
Figure B size(bytes): 1548412


## Final figures (side-by-side comparison set)

Run the previous three Java cells in order:
1. Pipeline generation for both input files
2. Final figure composition for both inputs
3. Orientation CSV comparison (tile-by-tile)

Figure A (C15D5P001):

![](_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_final_figure_java.png)

<img src="_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_final_figure_java.png" alt="Figure A" width="100%" />

Figure B (Picture1):

![](_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_final_figure_java.png)

<img src="_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_final_figure_java.png" alt="Figure B" width="100%" />

Direct link (Figure B): [_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_final_figure_java.png](_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_final_figure_java.png)